## Import

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"   # single-threaded so parallel jobs do not fight over cores

import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/waroonragwongsiri/Playground/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
SEED = 42
N_TRIALS = 30
INNER_K = 3
np.random.seed(SEED)
K, REPEATS = 5, 3
os.makedirs("weights_V5_1", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()  # 2213 rows (target must not be imputed)
print("cleaned:", df.shape, "| Income max:", f"{df['Income'].max():,.0f}", "| median:", f"{df['Income'].median():,.0f}")
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

cleaned: (2212, 28) | Income max: 162,397 | median: 51,371
features: 13 dims (all numeric)


## Feature Interaction

In [4]:
# Feature interaction hand-off (Feature Interaction section of regression.ipynb):
# Teenhome interacts with most spend features but is not in the 13-feature model set.
EXTRA = ["Teenhome"]
COLS_ALL = REG_FEATURES + EXTRA
X_all = df[COLS_ALL].values.astype(float)

def top_pairs(csv_path, cols, k, must_include=None, exclude=None):
    """Top-k significant (q < 0.05) pairs by effect size where both features are in `cols`."""
    t = pd.read_csv(csv_path)
    t = t[(t.q < 0.05) & t.A.isin(cols) & t.B.isin(cols)]
    if must_include is not None:
        t = t[t.A.eq(must_include) | t.B.eq(must_include)]
    if exclude is not None:
        t = t[~(t.A.eq(exclude) | t.B.eq(exclude))]
    t = t.sort_values("effect", ascending=False)
    return list(zip(t.A.head(k), t.B.head(k)))

PAIRS_13 = top_pairs("results/interaction_reg.csv", REG_FEATURES, 4)
PAIRS_T = top_pairs("results/interaction_reg.csv", COLS_ALL, 5, must_include="Teenhome")
print("pairs (13 features):", PAIRS_13); print("pairs (with Teenhome):", PAIRS_T)
SETS = {
    "13 features":                    dict(cols=REG_FEATURES, pairs=[]),
    "+ Teenhome (14)":                dict(cols=COLS_ALL, pairs=[]),
    "13 features + 4 pair nets":      dict(cols=REG_FEATURES, pairs=PAIRS_13),
    "+ Teenhome + 5 Teenhome pairs":  dict(cols=COLS_ALL, pairs=PAIRS_T),
}

def prep_split(tr_idx, te_idx, set_name):
    cols = SETS[set_name]["cols"]
    ci = [COLS_ALL.index(c) for c in cols]
    sc = StandardScaler().fit(X_all[tr_idx][:, ci])
    return sc.transform(X_all[tr_idx][:, ci]), sc.transform(X_all[te_idx][:, ci]), y_raw[tr_idx], y_raw[te_idx]

idx_all = np.arange(len(y_raw))
tr_idx, te_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED)
for s_ in SETS:
    a_, b_, _, _ = prep_split(tr_idx, te_idx, s_)
    print(f"{s_:32s} train {a_.shape} test {b_.shape}")

pairs (13 features): [('MntGoldProds', 'NumWebPurchases'), ('MntFruits', 'NumDealsPurchases'), ('MntFishProducts', 'NumWebPurchases'), ('NumWebPurchases', 'NumCatalogPurchases')]
pairs (with Teenhome): [('Teenhome', 'NumCatalogPurchases'), ('Teenhome', 'NumWebVisitsMonth'), ('Teenhome', 'MntWines'), ('Teenhome', 'MntMeatProducts'), ('Teenhome', 'NumStorePurchases')]
13 features                      train (1769, 13) test (443, 13)
+ Teenhome (14)                  train (1769, 14) test (443, 14)
13 features + 4 pair nets        train (1769, 13) test (443, 13)
+ Teenhome + 5 Teenhome pairs    train (1769, 14) test (443, 14)


In [5]:
# trees learn interactions on their own; pair subnets only exist in the NAM, so keep the two feature sets
SETS = {k: v for k, v in SETS.items() if not v["pairs"]}
print(list(SETS))

['13 features', '+ Teenhome (14)']


## Model

In [6]:
import lightgbm as lgb
SPECS = {
 "LightGBM": dict(tier="curriculum family",
    space=lambda t: dict(n=t.suggest_int("n", 100, 600, step=100), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         nl=t.suggest_int("nl", 4, 64), mcs=t.suggest_int("mcs", 5, 50),
                         sub=t.suggest_float("sub", 0.6, 1.0), col=t.suggest_float("col", 0.5, 1.0),
                         lam=t.suggest_float("lam", 1e-2, 10, log=True)),
    make=lambda p, s: lgb.LGBMRegressor(n_estimators=p["n"], learning_rate=p["lr"], num_leaves=p["nl"],
                         min_child_samples=p["mcs"], subsample=p["sub"], subsample_freq=1, colsample_bytree=p["col"],
                         reg_lambda=p["lam"], n_jobs=1, random_state=s, verbosity=-1)),
}

In [7]:
def inner_oof(spec, params, X, y, seed):
    """Out-of-fold predictions on the training rows (inner K-fold)."""
    oof = np.zeros(len(y))
    for a, b in KFold(INNER_K, shuffle=True, random_state=seed).split(X):
        est = spec["make"](params, seed)
        est.fit(X[a], y[a])
        oof[b] = est.predict(X[b])
    return oof

def tune(spec, X, y, seed):
    """Optuna (TPE) minimising RMSE of the inner out-of-fold predictions."""
    if spec.get("fixed"):
        return {}
    def objective(trial):
        return float(np.sqrt(mean_squared_error(y, inner_oof(spec, spec["space"](trial), X, y, seed))))
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=N_TRIALS)
    return spec["space"](optuna.trial.FixedTrial(study.best_params))

def fit_eval(spec, params, Xa, ya, Xb, yb, seed):
    est = spec["make"](params, seed)
    est.fit(Xa, ya)
    pred = est.predict(Xb)
    mse = mean_squared_error(yb, pred)
    return {"r2": r2_score(yb, pred), "rmse": float(np.sqrt(mse)), "mae": mean_absolute_error(yb, pred), "mse": mse}, pred

## Sanity Check

In [8]:
a, b = next(KFold(5, shuffle=True, random_state=SEED).split(X_all))
assert len(set(a) & set(b)) == 0, "train / eval rows overlap"
for sname, spec in SETS.items():
    Xa, Xb, ya, yb = prep_split(a, b, sname)
    assert Xa.shape[1] == Xb.shape[1] == len(spec["cols"])
    assert np.allclose(Xa.mean(0), 0, atol=1e-6) and np.allclose(Xa.std(0), 1, atol=1e-6)
    assert np.isfinite(Xa).all() and np.isfinite(Xb).all()
    print(f"{sname:32s} dims {Xa.shape[1]:3d} | pair subnets {len(spec['pairs'])} -> OK")

13 features                      dims  13 | pair subnets 0 -> OK
+ Teenhome (14)                  dims  14 | pair subnets 0 -> OK


## Training

In [9]:
spec = SPECS["LightGBM"]
cache = "weights_V5_1/_progress_regression_lightgbm.pkl"
state = pickle.load(open(cache, "rb")) if os.path.exists(cache) else {"rows": [], "scores": {}}
done = {(r["model"], r["repeat"], r["fold"]) for r in state["rows"]}
for rep in range(1, REPEATS + 1):
    seed_r = SEED + rep - 1               # same outer folds as V6_1 / V4_1
    for fold, (a, b) in enumerate(KFold(K, shuffle=True, random_state=seed_r).split(X_all), start=1):
        for sname in SETS:
            name = f"LightGBM | {sname}"
            if (name, rep, fold) in done:
                continue
            Xa, Xb, ya, yb = prep_split(a, b, sname)
            t0 = time.time()
            params = tune(spec, Xa, ya, seed_r * 100 + fold)
            row, s = fit_eval(spec, params, Xa, ya, Xb, yb, seed_r * 100 + fold)
            extra = dict(params=params)
            state["rows"].append(dict(model=name, tier="curriculum family", selection="inner validation", repeat=rep, fold=fold,
                                      n_test=int(len(yb)), **{k: round(float(v), 4) for k, v in row.items()},
                                      seconds=round(time.time() - t0, 1), **extra))
            state["scores"].setdefault(f"{name}|r{rep}", np.full(len(y_raw), np.nan))[b] = s
        pickle.dump(state, open(cache, "wb"))
        print(f"repeat {rep} fold {fold} done | rows so far {len(state['rows'])}", flush=True)

repeat 1 fold 1 done | rows so far 2


repeat 1 fold 2 done | rows so far 4


repeat 1 fold 3 done | rows so far 6


repeat 1 fold 4 done | rows so far 8


repeat 1 fold 5 done | rows so far 10


...

repeat 3 fold 1 done | rows so far 22


repeat 3 fold 2 done | rows so far 24


repeat 3 fold 3 done | rows so far 26


repeat 3 fold 4 done | rows so far 28


repeat 3 fold 5 done | rows so far 30


## K-Fold Summary

In [10]:
kdf = pd.DataFrame(state["rows"])
KEYS = ['r2', 'rmse', 'mae']
summary = {m: {k: {"mean": round(float(g[k].mean()), 4), "std": round(float(g[k].std()), 4)} for k in KEYS if k in g and g[k].notna().any()}
           for m, g in kdf.groupby("model")}
print(kdf.groupby("model")["r2"].agg(["mean", "std"]).sort_values("mean", ascending=False).round(4).to_string())
json.dump({"k": K, "repeats": REPEATS, "folds": state["rows"], "summary": summary},
          open("weights_V5_1/regression_lightgbm_kfold_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V5_1/regression_lightgbm_kfold_scores.npz", **state["scores"])

                              mean     std
model                                     
LightGBM | + Teenhome (14)  0.8263  0.0541
LightGBM | 13 features      0.8135  0.0644
